In [ ]:
# print("123")

123


In [21]:
import pandas as pd

from ingest import load_faq_data, build_index
from tqdm.auto import tqdm

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df_ground_truth = pd.read_csv("data/ground-truth.csv")

In [6]:
df_ground_truth.head()

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [7]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [8]:
ground_truth[10]

{'question': 'Why does sending a student question directly to a plain LLM usually give a bad answer for course-specific stuff?',
 'filename': '01-agentic-rag/lessons/03-rag.md'}

In [10]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [11]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.'},
 {'id': 'db78580409

In [12]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [16]:
q = ground_truth[0]

In [ ]:
doc_id = q['document']
doc_id

In [18]:
results = text_search(q['question'])
results

[{'id': '5e4ca64cb5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1 Homework',
  'question': 'Do homework answers need to match the options exactly?',
  'answer': 'Not always. If your numeric answer is close to one of the options, choose the closest option.\n\nSmall differences can come from:\n\n- Slightly different filtering.\n- Different dataset versions.\n- Floating-point or rounding differences.\n- Different model/provider behavior.\n\nIf your answer is far from every option, re-check the question, the dataset version, and the GitHub homework instructions.\n\nSee the general homework guidance: [homework logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/homework/).'},
 {'id': '93a3e8b98c',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why does a generic AI assistant generate Kestra flow YAML with properties that don't exist, and how can I avoid it?",
  'answer': 'A general AI (e.g., ChatGPT) isn\'t grounded in Kestra\'s plugi

In [ ]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

In [ ]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

In [ ]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

In [ ]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

In [ ]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)

In [ ]:
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
relevance = compute_relevance_total_text(ground_truth)

In [ ]:
relevance[:15]

In [ ]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
relevance_total = compute_relevance_total(ground_truth, text_search)

In [ ]:
sample = relevance_total[:15]

In [ ]:
sample

In [ ]:
14 / 15

In [ ]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

In [ ]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
hit_rate(relevance)

In [ ]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [ ]:
mrr(sample)

In [ ]:
mrr(relevance)

In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
evaluate(ground_truth, text_search)

In [ ]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
evaluate(ground_truth, text_search_v2)

In [ ]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

In [ ]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

In [ ]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)